# YOLOv8 Object Detection Training — Durian Disease Detection

## Mục tiêu

Huấn luyện YOLOv8 để phát hiện bounding box các bệnh trên lá sầu riêng.

**Dataset:** Annotations từ GroundingDINO+SAM pipeline (`/kaggle/working/data/detect_yolo/`)

**GPU:** NVIDIA Tesla T4 (16GB VRAM) — Kaggle

## Workflow

```
Cell 1: Setup + Verify Dataset
Cell 2: Configure Model & Hyperparameters
Cell 3: Train YOLOv8
Cell 4: Evaluate on Test Set
Cell 5: Inference Demo
```

**Model variants:** YOLOv8n (nano) → YOLOv8s (small) → YOLOv8m (medium)

**Start with YOLOv8n for fast iteration, then scale up.**


In [1]:
# ============================================================
# Cell 1 — Setup + Dataset Verification
# ============================================================

import os
import sys
import shutil
from pathlib import Path
import torch
import yaml
from glob import glob

print("=" * 60)
print("YOLOv8 Object Detection Training — Durian Disease")
print("=" * 60)

# Paths
YOLO_DATA_ROOT = Path("/kaggle/input/datasets/tranmanh1312/durian-yolo-annotations/detect_yolo")
YOLO_DATASET_YAML = YOLO_DATA_ROOT / "dataset.yaml"
OUTPUT_ROOT = Path("/kaggle/working/results/detection")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# --- Fix dataset.yaml path to match actual data location ---
if YOLO_DATASET_YAML.exists():
    with open(YOLO_DATASET_YAML, "r") as f:
        dataset_cfg = yaml.safe_load(f)
    # Update path field to actual data location
    if dataset_cfg.get("path") != str(YOLO_DATA_ROOT):
        print(f"[FIX] Updating dataset.yaml path:")
        print(f"  OLD: {dataset_cfg.get('path')}")
        print(f"  NEW: {YOLO_DATA_ROOT}")
        dataset_cfg["path"] = str(YOLO_DATA_ROOT)
        fixed_yaml = OUTPUT_ROOT / "dataset_fixed.yaml"
        with open(fixed_yaml, "w") as f:
            yaml.dump(dataset_cfg, f)
        YOLO_DATASET_YAML = fixed_yaml
        print(f"[FIX] Using fixed dataset.yaml: {YOLO_DATASET_YAML}")

print(f"\nTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify dataset.yaml exists
if not YOLO_DATASET_YAML.exists():
    raise FileNotFoundError(
        f"dataset.yaml not found at {YOLO_DATASET_YAML}\n"
        f"Run object-detection-final.ipynb Cell 8 first to generate YOLO dataset."
    )

print(f"\n>>> Dataset YAML: {YOLO_DATASET_YAML}")

# Load and display dataset.yaml
with open(YOLO_DATASET_YAML, "r") as f:
    dataset_cfg = yaml.safe_load(f)

print(f"\n--- dataset.yaml content ---")
for key, value in dataset_cfg.items():
    print(f"  {key}: {value}")

num_classes = dataset_cfg.get("nc", 0)
class_names = dataset_cfg.get("names", {})
print(f"\n  Number of classes: {num_classes}")
for idx, name in class_names.items():
    print(f"    [{idx}] {name}")

# Verify images/labels directories
splits = ["train", "val", "test"]
print(f"\n--- Dataset Split Verification ---")
total_images = 0
total_labels = 0
for split in splits:
    img_dir = YOLO_DATA_ROOT / "images" / split
    lbl_dir = YOLO_DATA_ROOT / "labels" / split
    if img_dir.exists():
        n_images = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        total_images += n_images
    else:
        n_images = 0
    if lbl_dir.exists():
        n_labels = len(list(lbl_dir.glob("*.txt")))
        total_labels += n_labels
    else:
        n_labels = 0
    
    status = "OK" if n_images > 0 else "EMPTY"
    print(f"  {split:6s}: {n_images:4d} images, {n_labels:4d} labels [{status}]")

print(f"\n  TOTAL: {total_images} images, {total_labels} labels")

if total_images == 0:
    raise RuntimeError("No images found. Run object-detection-final.ipynb Cell 7-8 first!")

# Count boxes per split
print(f"\n--- Per-split Box Count ---")
for split in splits:
    lbl_dir = YOLO_DATA_ROOT / "labels" / split
    if not lbl_dir.exists():
        continue
    total_boxes = 0
    for lbl_file in lbl_dir.glob("*.txt"):
        lines = lbl_file.read_text().strip().split("\n")
        total_boxes += len([l for l in lines if l.strip()])
    n_images = len(list((YOLO_DATA_ROOT / "images" / split).glob("*.*")))
    avg = total_boxes / max(n_images, 1)
    print(f"  {split:6s}: {total_boxes:4d} boxes, avg {avg:.2f} boxes/image ({n_images} images)")

# Show sample label file
sample_lbl = next((YOLO_DATA_ROOT / "labels" / "train").glob("*.txt"), None)
if sample_lbl:
    print(f"\n--- Sample label file ({sample_lbl.name}) ---")
    content = sample_lbl.read_text().strip()
    print(content if content else "(empty - no objects in this image)")

print("\n>>> Dataset verification complete!")


YOLOv8 Object Detection Training — Durian Disease

Torch version: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4
CUDA memory: 15.6 GB

>>> Dataset YAML: /kaggle/input/datasets/tranmanh1312/durian-yolo-annotations/detect_yolo/dataset.yaml

--- dataset.yaml content ---
  path: /kaggle/working/data/detect_yolo
  train: images/train
  val: images/val
  test: images/test
  nc: 2
  names: {0: 'ALGAL_LEAF_SPOT', 1: 'LEAF_BLIGHT'}

  Number of classes: 2
    [0] ALGAL_LEAF_SPOT
    [1] LEAF_BLIGHT

--- Dataset Split Verification ---
  train : 1008 images, 1008 labels [OK]
  val   :  143 images,  143 labels [OK]
  test  :  280 images,  280 labels [OK]

  TOTAL: 1431 images, 1431 labels

--- Per-split Box Count ---
  train : 2284 boxes, avg 2.27 boxes/image (1008 images)
  val   :  291 boxes, avg 2.03 boxes/image (143 images)
  test  :  607 boxes, avg 2.17 boxes/image (280 images)

--- Sample label file (to_label_3309.txt) ---
1 0.455357 0.723214 0.303571 0.508929

>>> Dataset verificat

# Cell 2 — Model & Training Configuration

## Model Selection

| Model | Params | mAP50 | VRAM (T4) | Speed | Recommended |
|-------|--------|-------|-------------|-------|-------------|
| YOLOv8n | 3.2M | 37.4% | ~2.5GB | Fastest | Start here |
| YOLOv8s | 11.2M | 44.9% | ~3.5GB | Fast | Production |
| YOLOv8m | 25.9M | 50.2% | ~5.5GB | Medium | High accuracy |

**Recommendation:** Start with `yolov8n`, iterate fast. Then scale to `yolov8s` or `yolov8m`.

## Hyperparameters

These settings are tuned for the T4 GPU and durian disease dataset:
- **imgsz=640**: Good balance of speed vs accuracy for 224x224 source images
- **epochs=100**: Sufficient for transfer learning (YOLOv8 converges fast)
- **patience=30**: Early stopping — stop if no improvement for 30 epochs
- **amp=True**: Mixed precision for 2x speed on T4
- **box_loss gain=7.5**: Standard YOLO box loss weight
- **cls_loss gain=0.5**: Classification loss weight (lower = trust box predictions more)

## Augmentations

YOLOv8 applies augmentations automatically. Key settings:
- **fliplr=0.5**: Horizontal flip (leaf orientation doesn't matter)
- **flipud=0.0**: NO vertical flip (natural orientation)
- **mosaic=1.0**: Full mosaic augmentation (combines 4 images)
- **mixup=0.1**: Light mixup for regularization
- **scale=0.5**: Scale variation up to 50%
- **degrees=10**: Small rotation (±10°)


In [2]:
# ============================================================
# Cell 2 — Training Configuration
# ============================================================

# ---- Model Selection ----
# Options: 'yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt'
MODEL_NAME = "yolov8n.pt"  # Start with nano for fast iteration

# ---- Training Hyperparameters ----
EPOCHS = 100
IMG_SIZE = 640       # Input resolution (higher = slower but more accurate)
BATCH_SIZE = 32      # T4 has 16GB VRAM, 32 is safe for nano/small
PATIENCE = 30        # Early stopping epochs
SAVE_PERIOD = 10     # Save checkpoint every N epochs

# ---- Optimizer ----
OPTIMIZER = "AdamW"  # AdamW or SGD
LR0 = 0.001          # Initial learning rate
LRF = 0.01           # Final learning rate (as fraction of LR0)
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005

# ---- Loss Weights ----
BOX_GAIN = 7.5
CLS_GAIN = 0.5
DFL_GAIN = 1.5

# ---- Augmentations ----
AUG_HSV_H = 0.015
AUG_HSV_S = 0.7
AUG_HSV_V = 0.4
AUG_DEGREES = 10.0
AUG_TRANSLATE = 0.1
AUG_SCALE = 0.5
AUG_SHEAR = 0.0
AUG_PERSPECTIVE = 0.0
AUG_FLIPUD = 0.0      # No vertical flip (natural orientation)
AUG_FLIPLR = 0.5      # Horizontal flip OK
AUG_MOSAIC = 1.0        # Mosaic (very effective)
AUG_MIXUP = 0.1        # Light mixup
AUG_COPYPASTE = 0.0

# ---- Project/Run Names ----
PROJECT_NAME = "yolo_durian_detection"
RUN_NAME = "yolov8n_v1"  # Change for each run

# Print configuration
print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"  Model:             {MODEL_NAME}")
print(f"  Dataset:           {YOLO_DATA_ROOT}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Image Size:         {IMG_SIZE}")
print(f"  Batch Size:         {BATCH_SIZE}")
print(f"  Patience:           {PATIENCE}")
print(f"  Optimizer:          {OPTIMIZER}")
print(f"  Learning Rate:     {LR0} (final: {LR0 * LRF})")
print(f"  Loss gains:        box={BOX_GAIN}, cls={CLS_GAIN}, dfl={DFL_GAIN}")
print(f"  AMP (mixed prec):  True")
print(f"  Project:           {PROJECT_NAME}")
print(f"  Run name:          {RUN_NAME}")
print(f"  Output:            {OUTPUT_ROOT / PROJECT_NAME / RUN_NAME}")
print("=" * 60)


TRAINING CONFIGURATION
  Model:             yolov8n.pt
  Dataset:           /kaggle/input/datasets/tranmanh1312/durian-yolo-annotations/detect_yolo
  Epochs:            100
  Image Size:         640
  Batch Size:         32
  Patience:           30
  Optimizer:          AdamW
  Learning Rate:     0.001 (final: 1e-05)
  Loss gains:        box=7.5, cls=0.5, dfl=1.5
  AMP (mixed prec):  True
  Project:           yolo_durian_detection
  Run name:          yolov8n_v1
  Output:            /kaggle/working/results/detection/yolo_durian_detection/yolov8n_v1


In [3]:
# ============================================================
# Cell 2.5 — Install Dependencies
# ============================================================
!pip install ultralytics -q
print("ultralytics installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23

# Cell 3 — Train YOLOv8

## Lưu ý quan trọng

1. **Cell 1 & 2 phải chạy trước** — dataset verification và configuration
2. **Kaggle T4**: Training ~2.5-3h cho 100 epochs YOLOv8n
3. **Checkpointing**: YOLOv8 tự động lưu `weights/best.pt` và `weights/last.pt`
4. **Metric chính**: mAP50 (mAP @ IoU=0.50) — càng cao càng tốt
5. **Early stopping**: Dừng sớm nếu val mAP50 không cải thiện trong 30 epochs

## Training Output

Sau khi train xong, kết quả sẽ có ở:
```
/kaggle/working/results/detection/yolo_durian_detection/yolov8n_v1/
├── weights/
│   ├── best.pt       ← Model tốt nhất (theo val mAP50)
│   └── last.pt       ← Model epoch cuối
├── results.csv       ← Training metrics per epoch
├── results.png      ← Training curves
├── args.yaml        ← Training arguments
└── ...
```


In [4]:
# ============================================================
# Cell 3 — Train YOLOv8
# ============================================================

from ultralytics import YOLO
import torch

print("=" * 60)
print("STARTING YOLOV8 TRAINING")
print("=" * 60)

# Load pretrained model
print(f"\nLoading {MODEL_NAME}...")
model = YOLO(MODEL_NAME)

# Train
print(f"\nTraining: {EPOCHS} epochs, batch={BATCH_SIZE}, imgsz={IMG_SIZE}")
print(f"Dataset: {YOLO_DATASET_YAML}")
print(f"Output: {OUTPUT_ROOT / PROJECT_NAME / RUN_NAME}")
print("-" * 60)

results = model.train(
    # Data
    data=str(YOLO_DATASET_YAML),
    # Model & Task
    task="detect",
    # Training params
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    save=True,
    save_period=SAVE_PERIOD,
    # Device
    device=0 if torch.cuda.is_available() else "cpu",
    # Output
    project=str(OUTPUT_ROOT / PROJECT_NAME),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
    # Mixed precision (AMP)
    amp=True,
    # Optimizer
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    # Loss weights
    box=BOX_GAIN,
    cls=CLS_GAIN,
    dfl=DFL_GAIN,
    # Augmentations
    hsv_h=AUG_HSV_H,
    hsv_s=AUG_HSV_S,
    hsv_v=AUG_HSV_V,
    degrees=AUG_DEGREES,
    translate=AUG_TRANSLATE,
    scale=AUG_SCALE,
    shear=AUG_SHEAR,
    perspective=AUG_PERSPECTIVE,
    flipud=AUG_FLIPUD,
    fliplr=AUG_FLIPLR,
    mosaic=AUG_MOSAIC,
    mixup=AUG_MIXUP,
    copy_paste=AUG_COPYPASTE,
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

# Best model path
best_model_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "weights" / "best.pt"
last_model_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "weights" / "last.pt"
print(f"\nBest model:  {best_model_path}")
print(f"Last model: {last_model_path}")

if best_model_path.exists():
    best_size_mb = best_model_path.stat().st_size / 1e6
    print(f"Best model size: {best_size_mb:.1f} MB")

# Show training summary
print("\n--- Training Results ---")
print(results)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
STARTING YOLOV8 TRAINING

Loading yolov8n.pt...

Training: 100 epochs, batch=32, imgsz=640
Dataset: /kaggle/input/datasets/tranmanh1312/durian-yolo-annotations/detect_yolo/dataset.yaml
Output: /kaggle/working/results/detection/yolo_durian_detection/yolov8n_v1
------------------------------------------------------------
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix

RuntimeError: Dataset '/kaggle/input/datasets/tranmanh1312/durian-yolo-annotations/detect_yolo/dataset.yaml' error ❌ Dataset '/kaggle/input/datasets/tranmanh1312/durian-yolo-annotations/detect_yolo/dataset.yaml' images not found, missing path '/kaggle/working/data/detect_yolo/images/val'
Note dataset download directory is '/kaggle/working/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

# Cell 4 — Evaluate on Test Set

## Metrics

| Metric | Mô tả | Target |
|--------|--------|--------|
| **mAP50** | mAP @ IoU=0.50 | > 0.50 |
| **mAP50-95** | mAP trung bình IoU 0.50-0.95 | > 0.30 |
| **Precision** | Tỷ lệ dự đoán đúng | > 0.70 |
| **Recall** | Tỷ lệ phát hiện đúng | > 0.60 |
| **F1** | Harmonic mean của P và R | > 0.65 |

## Per-Class mAP

Mỗi class nên có mAP50 riêng. Nếu class nào có mAP50 thấp (<0.20), có thể:
- Dataset có ít annotations cho class đó
- Annotations quality kém
- Model gặp khó khăn với class đó


In [ ]:
# ============================================================
# Cell 4 — Evaluate on Test Set
# ============================================================

from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load best model
best_model_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "weights" / "best.pt"
if not best_model_path.exists():
    raise FileNotFoundError(f"Best model not found: {best_model_path}")

print("=" * 60)
print("EVALUATING ON TEST SET")
print("=" * 60)
print(f"Model: {best_model_path}")

model = YOLO(str(best_model_path))

# Evaluate on test set
print(f"\nEvaluating on test split...")
metrics = model.val(
    data=str(YOLO_DATASET_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    save_json=True,
    plots=True,
    verbose=True,
)

# Print overall metrics
print("\n" + "=" * 60)
print("OVERALL TEST METRICS")
print("=" * 60)
print(f"  mAP50 (box):       {metrics.box.map50:.4f}")
print(f"  mAP50-95 (box):    {metrics.box.map:.4f}")
print(f"  Precision (box):    {metrics.box.mp:.4f}")
print(f"  Recall (box):       {metrics.box.mr:.4f}")

# Per-class metrics
print("\n--- Per-Class mAP50 ---")
per_class_map50 = metrics.box.ap50
per_class_map = metrics.box.ap
for idx, name in class_names.items():
    map50 = per_class_map50[idx] if idx < len(per_class_map50) else 0.0
    map = per_class_map[idx] if idx < len(per_class_map) else 0.0
    bar = "#" * int(map50 * 20)
    print(f"  [{idx}] {name:30s} mAP50={map50:.4f}  mAP={map:.4f}  {bar}")

# Per-class precision/recall
print("\n--- Per-Class Precision/Recall ---")
per_class_p = getattr(metrics.box, "p", None)
per_class_r = getattr(metrics.box, "r", None)
if per_class_p is not None and per_class_r is not None:
    for idx, name in class_names.items():
        p = per_class_p[idx] if idx < len(per_class_p) else 0.0
        r = per_class_r[idx] if idx < len(per_class_r) else 0.0
        f1 = 2 * p * r / (p + r + 1e-8)
        print(f"  [{idx}] {name:30s} P={p:.4f}  R={r:.4f}  F1={f1:.4f}")
else:
    print("  (per-class P/R not available)")
    per_class_p = [0.0] * num_classes
    per_class_r = [0.0] * num_classes

print("\n--- Ground Truth Box Distribution ---")
from collections import defaultdict
gt_counts = defaultdict(int)
for split in ["train", "val", "test"]:
    lbl_dir = YOLO_DATA_ROOT / "labels" / split
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().split("\n"):
            if line.strip():
                cls_id = int(line.strip().split()[0])
                gt_counts[cls_id] += 1

for idx, name in class_names.items():
    count = gt_counts.get(idx, 0)
    print(f"  [{idx}] {name:30s} {count:5d} GT boxes")

# Save metrics to CSV
metrics_csv = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "test_metrics.csv"
rows = []
for idx, name in class_names.items():
    map50 = per_class_map50[idx] if idx < len(per_class_map50) else 0.0
    map = per_class_map[idx] if idx < len(per_class_map) else 0.0
    p = per_class_p[idx] if idx < len(per_class_p) else 0.0
    r = per_class_r[idx] if idx < len(per_class_r) else 0.0
    rows.append({
        "class_id": idx,
        "class_name": name,
        "GT_boxes": gt_counts.get(idx, 0),
        "mAP50": map50,
        "mAP": map,
        "Precision": p,
        "Recall": r,
        "F1": 2*p*r/(p+r+1e-8)
    })
df = pd.DataFrame(rows)
df.to_csv(metrics_csv, index=False)
print(f"\nMetrics saved: {metrics_csv}")
print(df.to_string(index=False))

# Plot per-class bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

class_labels = [class_names[i] for i in range(len(class_names))]
map50_vals = [per_class_map50[i] if i < len(per_class_map50) else 0 for i in range(len(class_names))]
p_vals = [per_class_p[i] if i < len(per_class_p) else 0 for i in range(len(class_names))]
r_vals = [per_class_r[i] if i < len(per_class_r) else 0 for i in range(len(class_names))]

colors = ["#4CAF50", "#FF9800", "#2196F3", "#F44336", "#9C27B0"]

x = np.arange(len(class_labels))
w = 0.25

axes[0].bar(x, map50_vals, color=colors[:len(x)], alpha=0.8, edgecolor="black")
axes[0].set_xticks(x)
axes[0].set_xticklabels(class_labels, rotation=30, ha="right")
axes[0].set_ylabel("mAP50")
axes[0].set_title("Per-Class mAP50 (higher is better)")
axes[0].axhline(y=np.mean(map50_vals), color="red", linestyle="--", label=f"Mean={np.mean(map50_vals):.3f}")
axes[0].legend()
axes[0].set_ylim(0, 1.0)

axes[1].bar(x, p_vals, color=colors[:len(x)], alpha=0.8, edgecolor="black")
axes[1].set_xticks(x)
axes[1].set_xticklabels(class_labels, rotation=30, ha="right")
axes[1].set_ylabel("Precision")
axes[1].set_title("Per-Class Precision (higher is better)")
axes[1].axhline(y=np.mean(p_vals), color="red", linestyle="--", label=f"Mean={np.mean(p_vals):.3f}")
axes[1].legend()
axes[1].set_ylim(0, 1.0)

axes[2].bar(x, r_vals, color=colors[:len(x)], alpha=0.8, edgecolor="black")
axes[2].set_xticks(x)
axes[2].set_xticklabels(class_labels, rotation=30, ha="right")
axes[2].set_ylabel("Recall")
axes[2].set_title("Per-Class Recall (higher is better)")
axes[2].axhline(y=np.mean(r_vals), color="red", linestyle="--", label=f"Mean={np.mean(r_vals):.3f}")
axes[2].legend()
axes[2].set_ylim(0, 1.0)

plt.tight_layout()
chart_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "per_class_metrics.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nChart saved: {chart_path}")


# Cell 5 — Inference Demo

## Single Image Detection

Load best model và chạy inference trên ảnh test để visualize kết quả.

## Interpretation

- **Bounding box**: Vùng phát hiện bệnh
- **Confidence**: Độ tự tin của model (0-1)
- **Class color**: Mỗi class có màu riêng

| Class | Color |
|-------|-------|
| ALGAL_LEAF_SPOT | Green |
| ALLOCARIDARA_ATTACK | Orange |
| LEAF_BLIGHT | Red |
| PHOMOPSIS_LEAF_SPOT | Purple |


In [ ]:
# ============================================================
# Cell 5 — Inference Demo
# ============================================================

from ultralytics import YOLO
# Ground truth box distribution
from collections import defaultdict
gt_counts = defaultdict(int)
for split in ["train", "val", "test"]:
    lbl_dir = YOLO_DATA_ROOT / "labels" / split
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().split("\n"):
            if line.strip():
                cls_id = int(line.strip().split()[0])
                gt_counts[cls_id] += 1

import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import random

# Load best model
best_model_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "weights" / "best.pt"
if not best_model_path.exists():
    print(f"Best model not found at {best_model_path}")
    print("Training may not have completed. Using last.pt instead.")
    best_model_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "weights" / "last.pt"

model = YOLO(str(best_model_path))
print(f"Loaded model: {best_model_path}")

# Class colors (BGR for CV2, RGB for matplotlib)
CLASS_COLORS = {
    0: (0, 200, 0),       # ALGAL_LEAF_SPOT — Green
    1: (0, 0, 255),       # LEAF_BLIGHT — Red
}

def display_detection(img_path, model, conf=0.25):
    """Run detection and display results."""
    results = model(img_path, conf=conf, imgsz=IMG_SIZE, verbose=False)
    result = results[0]
    
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    boxes = result.boxes
    n_dets = len(boxes)
    
    for box in boxes:
        xyxy = box.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = [int(v) for v in xyxy]
        cls_id = int(box.cls[0].cpu().item())
        conf_score = float(box.conf[0].cpu().item())
        class_name = class_names.get(cls_id, f"Class_{cls_id}")
        
        color = CLASS_COLORS.get(cls_id, (128, 128, 128))
        
        # Draw box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        
        # Draw label
        label = f"{class_name} {conf_score:.2f}"
        label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(img, (x1, y1 - label_size[1] - 6), (x1 + label_size[0], y1), color, -1)
        cv2.putText(img, label, (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img_rgb, n_dets, result

# Get test images
test_img_dir = YOLO_DATA_ROOT / "images" / "test"
test_images = sorted(test_img_dir.glob("*.jpg")) + sorted(test_img_dir.glob("*.png"))

print(f"Found {len(test_images)} test images")

# Show N random samples
N_SAMPLES = 8
sample_images = random.sample(test_images, min(N_SAMPLES, len(test_images)))

n_cols = 4
n_rows = (len(sample_images) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

total_detections = 0
for ax, img_path in zip(axes.flat, sample_images):
    img_rgb, n_dets, _ = display_detection(img_path, model, conf=0.15)
    total_detections += n_dets
    ax.imshow(img_rgb)
    ax.set_title(f"{img_path.name}\n{n_dets} detections", fontsize=9)
    ax.axis("off")

# Hide unused subplots
for ax in axes.flat[len(sample_images):]:
    ax.axis("off")

plt.suptitle(f"YOLOv8 Detection Results — {N_SAMPLES} Random Test Samples\n"
              f"Total: {total_detections} detections in {len(sample_images)} images",
              fontsize=14, fontweight="bold")
plt.tight_layout()
demo_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "detection_demo.png"
plt.savefig(demo_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nDemo saved: {demo_path}")

# Run detection on ALL test images and collect stats
print("\n--- Full Test Set Detection Stats ---")
detection_counts = {i: 0 for i in range(num_classes)}
images_with_det = 0
images_no_det = 0

for img_path in test_images:
    results = model(img_path, conf=0.15, verbose=False)
    boxes = results[0].boxes
    if len(boxes) > 0:
        images_with_det += 1
        for box in boxes:
            cls_id = int(box.cls[0].cpu().item())
            detection_counts[cls_id] += 1
    else:
        images_no_det += 1

print(f"\n  Images with detections:  {images_with_det}/{len(test_images)} ({images_with_det/len(test_images):.1%})")
print(f"  Images with NO detections: {images_no_det}/{len(test_images)} ({images_no_det/len(test_images):.1%})")
print(f"\n  Detection count per class:")
for idx, name in class_names.items():
    count = detection_counts.get(idx, 0)
    gt = gt_counts.get(idx, 0)
    print(f"    [{idx}] {name:30s} {count:4d} detections (GT: {gt})")

print("\n" + "=" * 60)
print("ALL DONE!")
print("=" * 60)
print(f"\nModel checkpoint: {best_model_path}")
print(f"Training output:  {OUTPUT_ROOT / PROJECT_NAME / RUN_NAME}")
print(f"\nTo export model:")
print(f"  model.export(format='onnx')  # ONNX (cross-platform)")
print(f"  model.export(format='torchscript')  # TorchScript")


# Cell 6 — Quick Resume / Retrain with Different Settings

## Khi nào cần retrain?

- Chạy lại với model lớn hơn (yolov8s, yolov8m)
- Thay đổi hyperparameters (imgsz, batch size)
- Dùng data augmentation khác
- Train thêm epochs

## Cách resume

Thay đổi `RUN_NAME` và chạy lại Cell 2 → Cell 3.
YOLOv8 sẽ train từ đầu với cùng dataset.

## Model Scaling Guide

| Model | Params | Batch | imgsz | Est. Time (T4) |
|-------|--------|-------|-------|-----------------|
| YOLOv8n | 3.2M | 32 | 640 | ~2.5h / 100 epochs |
| YOLOv8s | 11.2M | 24 | 640 | ~4h / 100 epochs |
| YOLOv8m | 25.9M | 16 | 640 | ~6h / 100 epochs |
| YOLOv8n | 3.2M | 32 | 800 | ~3.5h / 100 epochs |

**Batch size giảm nếu OOM.**

In [ ]:
# ============================================================
# Cell 13 — Package Outputs for Download
# ============================================================
# Tổng hợp tất cả outputs cần tải về máy local
# File .zip sẽ nằm ở /kaggle/working/ → tải từ Kaggle Output

import zipfile
import shutil
from pathlib import Path

run_dir = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME
download_dir = Path("/kaggle/working/download")
download_dir.mkdir(parents=True, exist_ok=True)

zip_path = download_dir / f"{PROJECT_NAME}_{RUN_NAME}_outputs.zip"

# Files/folders cần đóng gói
items_to_zip = [
    # Model weights (cần nhất cho web demo)
    (run_dir / "weights" / "best.pt",         "weights/best.pt"),
    (run_dir / "weights" / "last.pt",         "weights/last.pt"),
    # Exported formats
    (run_dir / "weights" / "best.onnx",       "weights/best.onnx"),
    (run_dir / "weights" / "best.torchscript","weights/best.torchscript"),
    # Metrics
    (run_dir / "test_metrics.csv",            "test_metrics.csv"),
    (run_dir / "training_summary.json",       "training_summary.json"),
    # Dataset config (đã fix path)
    (OUTPUT_ROOT / "dataset_fixed.yaml",       "dataset_fixed.yaml"),
    # Plots
    (run_dir / "confusion_matrix.png",        "plots/confusion_matrix.png"),
    (run_dir / "PR_curve.png",                "plots/PR_curve.png"),
    (run_dir / "results.png",                 "plots/results.png"),
    (run_dir / "val_batch0_pred.jpg",         "plots/val_batch0_pred.jpg"),
]

print("=" * 60)
print("PACKAGING OUTPUTS")
print("=" * 60)
print(f"\nRun directory:  {run_dir}")
print(f"Output zip:    {zip_path}\n")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for src_path, arc_name in items_to_zip:
        if src_path.exists():
            zf.write(src_path, arc_name)
            size_kb = src_path.stat().st_size / 1024
            print(f"  [OK] {arc_name:<45s} ({size_kb:.1f} KB)")
        else:
            print(f"  [--] {arc_name:<45s} (not found, skipped)")

print(f"\n✅ ZIP created: {zip_path}")
print(f"   Size: {zip_path.stat().st_size / 1024 / 1024:.1f} MB")
print("\n📦 Để tải về:")
print("   1. Mở tab 'Output' bên trái giao diện Kaggle")
print("   2. Tìm file .zip trong thư mục /kaggle/working/download/")
print("   3. Nhấn Download")


In [ ]:
# ============================================================
# Cell 6 — Export & Summary
# ============================================================

import torch
from ultralytics import YOLO
import json
from pathlib import Path

def _load_metrics_from_csv(run_dir):
    """Load metrics from test_metrics.csv saved by Cell 4."""
    csv_path = run_dir / "test_metrics.csv"
    if csv_path.exists():
        import pandas as pd
        df = pd.read_csv(csv_path)
        return {
            "mAP50": float(df["mAP50"].mean()) if "mAP50" in df.columns else 0.0,
            "mAP50-95": float(df["mAP"].mean()) if "mAP" in df.columns else 0.0,
            "Precision": float(df["Precision"].mean()) if "Precision" in df.columns else 0.0,
            "Recall": float(df["Recall"].mean()) if "Recall" in df.columns else 0.0,
        }
    return {"mAP50": 0.0, "mAP50-95": 0.0, "Precision": 0.0, "Recall": 0.0}



best_model_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_model_path))

print("=" * 60)
print("EXPORT MODEL")
print("=" * 60)

# Export to ONNX
print("\n>>> Exporting to ONNX...")
onnx_path = model.export(format="onnx", imgsz=IMG_SIZE)
print(f"    ONNX saved: {onnx_path}")

# Export to TorchScript
print("\n>>> Exporting to TorchScript...")
ts_path = model.export(format="torchscript", imgsz=IMG_SIZE)
print(f"    TorchScript saved: {ts_path}")

# Save training summary
summary = {
    "model": MODEL_NAME,
    "dataset_yaml": str(YOLO_DATASET_YAML),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "imgsz": IMG_SIZE,
    "num_classes": num_classes,
    "class_names": {str(k): v for k, v in class_names.items()},
    "best_model_path": str(best_model_path),
    "onnx_path": str(onnx_path),
    "torchscript_path": str(ts_path),
    "training_output": str(OUTPUT_ROOT / PROJECT_NAME / RUN_NAME),
    "metrics": _load_metrics_from_csv(OUTPUT_ROOT / PROJECT_NAME / RUN_NAME),
}

summary_path = OUTPUT_ROOT / PROJECT_NAME / RUN_NAME / "training_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"\nSummary saved: {summary_path}")

print("\n" + "=" * 60)
print("TRAINING PIPELINE COMPLETE")
print("=" * 60)
print(f"""
Model:        {MODEL_NAME}
Best weights: {best_model_path}
ONNX:         {onnx_path}
TorchScript:  {ts_path}

Metrics (test set):
  mAP50:        {summary["metrics"]["mAP50"]:.4f}
  mAP50-95:     {summary["metrics"]["mAP50-95"]:.4f}
  Precision:    {summary["metrics"]["Precision"]:.4f}
  Recall:       {summary["metrics"]["Recall"]:.4f}

Output directory:
  {OUTPUT_ROOT / PROJECT_NAME / RUN_NAME}
""")

print("\nNext steps:")
print("  1. Download weights from Kaggle output")
print("  2. Place best.pt in models/detection/yolov8n_disease/weights/")
print("  3. Update demo/app.py to include detection mode")
print("  4. Integrate with Flask web app")
